In [ ]:
import re

REGION_RE = re.compile(r"^\s*(\d+)x(\d+)\s*:\s*(.*)$")
SHAPE_HDR_RE = re.compile(r"^\s*(\d+)\s*:\s*$")


def parse_input(text: str):
    lines = [ln.rstrip("\n") for ln in text.splitlines()]

    first_region = None
    for i, ln in enumerate(lines):
        if REGION_RE.match(ln):
            first_region = i
            break
    if first_region is None:
        raise ValueError("No region lines found (expected 'WxH: ...').")

    shape_lines = lines[:first_region]
    region_lines = [ln for ln in lines[first_region:] if ln.strip()]

    shapes = {}
    i = 0
    while i < len(shape_lines):
        ln = shape_lines[i].strip()
        if not ln:
            i += 1
            continue
        m = SHAPE_HDR_RE.match(ln)
        if not m:
            raise ValueError(f"Bad shape header: {shape_lines[i]!r}")
        idx = int(m.group(1))
        i += 1

        grid = []
        while i < len(shape_lines) and shape_lines[i].strip() != "":
            grid.append(shape_lines[i])
            i += 1
        while i < len(shape_lines) and shape_lines[i].strip() == "":
            i += 1

        cells = []
        for y, row in enumerate(grid):
            for x, ch in enumerate(row):
                if ch == "#":
                    cells.append((x, y))
        if not cells:
            raise ValueError(f"Shape {idx} has no # cells.")
        shapes[idx] = cells

    n = max(shapes) + 1
    shape_list = [None] * n
    for k, v in shapes.items():
        shape_list[k] = v
    if any(v is None for v in shape_list):
        raise ValueError("Shape indices must be contiguous 0..max_index.")

    regions = []
    for ln in region_lines:
        m = REGION_RE.match(ln)
        if not m:
            continue
        w, h = int(m.group(1)), int(m.group(2))
        counts = list(map(int, m.group(3).split())) if m.group(3).strip() else []
        regions.append((w, h, counts))

    return shape_list, regions


def orientations(cells):
    seen = set()
    out = []
    for flip in (False, True):
        for rot in range(4):
            tr = []
            for x, y in cells:
                xx, yy = (-x, y) if flip else (x, y)
                for _ in range(rot):
                    xx, yy = yy, -xx
                tr.append((xx, yy))

            minx = min(x for x, _ in tr)
            miny = min(y for _, y in tr)
            norm = tuple(sorted((x - minx, y - miny) for x, y in tr))
            if norm in seen:
                continue
            seen.add(norm)
            maxx = max(x for x, _ in norm)
            maxy = max(y for _, y in norm)
            out.append((norm, maxx + 1, maxy + 1))
    return out


def precompute_for_size(W, H, shape_orients):
 
    n_shapes = len(shape_orients)
    n_cells = W * H

    placements_mask = []
    placements_cells = []
    areas = []

    for orients in shape_orients:
        area = len(orients[0][0])
        areas.append(area)
        masks = {}
        for cells, w, h in orients:
            if w > W or h > H:
                continue
            base = 0
            base_cells = []
            for x, y in cells:
                idx = y * W + x
                base |= 1 << idx
                base_cells.append(idx)

            for y0 in range(H - h + 1):
                row_shift = y0 * W
                for x0 in range(W - w + 1):
                    shift = row_shift + x0
                    m = base << shift
                    if m not in masks:
                        masks[m] = True

        plist = list(masks.keys())
        placements_mask.append(plist)
        pcells = []
        for m in plist:
            cells = []
            mm = m
            while mm:
                lsb = mm & -mm
                cells.append(lsb.bit_length() - 1)
                mm ^= lsb
            pcells.append(tuple(cells))
        placements_cells.append(pcells)

    cover = [[0] * n_cells for _ in range(n_shapes)]
    for s in range(n_shapes):
        for j, cells in enumerate(placements_cells[s]):
            bit = 1 << j
            for c in cells:
                cover[s][c] |= bit

    all_ones = [(1 << len(placements_mask[s])) - 1 for s in range(n_shapes)]
    full_board_mask = (1 << (W * H)) - 1
    return placements_mask, placements_cells, cover, all_ones, areas, full_board_mask


def can_fit_region_fast(W, H, counts, pre):
    placements_mask, placements_cells, cover, all_ones, areas, full_board_mask = pre
    n = len(placements_mask)
    counts = (counts + [0] * (n - len(counts)))[:n]

    total_area = sum(counts[i] * areas[i] for i in range(n))
    if total_area > W * H:
        return False
    for i, c in enumerate(counts):
        if c > 0 and not placements_mask[i]:
            return False

    SHIFT = 12
    MASK = (1 << SHIFT) - 1
    shifts = [SHIFT * i for i in range(n)]
    ckey0 = 0
    for i, c in enumerate(counts):
        if c > MASK:
            raise ValueError("Count too large; increase SHIFT.")
        ckey0 |= c << shifts[i]

    av0 = tuple(all_ones)
    memo = {}

    def dfs(occ_mask: int, avails: tuple[int, ...], ckey: int, remaining_area: int) -> bool:
        if ckey == 0:
            return True

        # enough empty cells?
        free = full_board_mask ^ occ_mask
        if remaining_area > free.bit_count():
            return False

        key = (occ_mask, ckey)
        if key in memo:
            return memo[key]

        best_i = -1
        best_opts = 10**18
        for i in range(n):
            need = (ckey >> shifts[i]) & MASK
            if need == 0:
                continue
            opts = avails[i].bit_count()
            if opts < need:
                memo[key] = False
                return False
            if opts < best_opts:
                best_opts = opts
                best_i = i
                if opts <= 1:
                    break

        ai = avails[best_i]
        if ai == 0:
            memo[key] = False
            return False

        while ai:
            lsb = ai & -ai
            j = lsb.bit_length() - 1
            ai ^= lsb

            pmask = placements_mask[best_i][j]
            if pmask & occ_mask:
                continue  
            new_av = list(avails)
            for t in range(n):
                invalid = 0
                for c in placements_cells[best_i][j]:
                    invalid |= cover[t][c]
                new_av[t] = new_av[t] & (~invalid & all_ones[t])

            new_ckey = ckey - (1 << shifts[best_i])
            if dfs(occ_mask | pmask, tuple(new_av), new_ckey, remaining_area - areas[best_i]):
                memo[key] = True
                return True

        memo[key] = False
        return False

    return dfs(0, av0, ckey0, total_area)


def solve_file():
    with open("input.txt", "r", encoding="utf-8") as f:
        shapes, regions = parse_input(f.read())

    shape_orients = [orientations(cells) for cells in shapes]

    cache = {} 
    ok = 0
    for W, H, counts in regions:
        key = (W, H)
        pre = cache.get(key)
        if pre is None:
            pre = precompute_for_size(W, H, shape_orients)
            cache[key] = pre
        if can_fit_region_fast(W, H, counts, pre):
            ok += 1
    return ok



print(solve_file())


593
